# ✈️ Orquestador ELT – Flights & Airports

Este notebook ejecuta y documenta el flujo **Extract → Load → Transform (ELT)** del proyecto.

## 🎯 Objetivos
- **Extract:** consultar la API de AviationStack y persistir crudos en **Parquet**.
- **Load:** cargar crudos a **Delta Lake** (control de esquema e histórico).
- **Transform:** aplanar, limpiar y enriquecer datos listos para análisis.

## 🔧 Configuración inicial

In [ ]:
# Asegurar imports y rutas
import os
import pandas as pd
from deltalake import DeltaTable
from deltalake.exceptions import TableNotFoundError
from pathlib import Path
import configparser

# --- Configuración de las rutas ---
script_dir = Path(os.getcwd())
base_path = script_dir.parent
config_path = base_path / "pipeline.config"

config = configparser.ConfigParser()
config.read(config_path)

DATA_LAKE_PATH = base_path / config.get("paths", "data_lake", fallback="data_lake")
FLIGHTS_RAW_PARQUET = DATA_LAKE_PATH / "flights" / "processed" / "flights_raw.parquet"
AIRPORTS_RAW_PARQUET = DATA_LAKE_PATH / "airports" / "processed" / "airports_raw.parquet"
FLIGHTS_DELTA_PATH = DATA_LAKE_PATH / "flights" / "delta"
AIRPORTS_DELTA_PATH = DATA_LAKE_PATH / "airports" / "delta"
PROCESSED_ENRICHED_PATH = DATA_LAKE_PATH / "flights" / "processed_enriched"
AGG_PATH = DATA_LAKE_PATH / "flights" / "agg_by_status"

print("✅ Rutas del proyecto configuradas con éxito.")


✅ Rutas de Delta Lake verificadas.
Ruta Data Lake: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake
Ruta Vuelos Delta: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\delta
Ruta Aeropuertos Delta: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\airports\delta
Ruta Enriquecidos: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\processed_enriched
Ruta Agregados: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\agg_by_status


## 1️⃣ Extract

Demostración del Flujo Incremental y Full

## Paso 1: Configuración Inicial ⚙️
Abre el archivo pipeline.config y verifica la sección [state]. La fecha de last_flights_run es el punto de partida para el proceso incremental. Este es el estado inicial de tu pipeline.

* [state]
last_flights_run = '2025-01-01'

## Paso 2: Extracción y Carga Full (Aeropuertos) ✈️
Ejecuta el script extract.py.

Este script se conectará a la API y extraerá los datos de vuelos para la fecha actual y los datos de aeropuertos completos (full).

Luego, guardará ambos conjuntos de datos como archivos .parquet en sus respectivos directorios.

### 📝 Aclaración Estrategia de Extracción Mejorada: Resiliencia del Pipeline

El script de extracción (extract.py) ahora implementa una estrategia de triple fallback en la extracción de datos de vuelos:

Intento principal (Incremental): Se realiza una llamada a la API para obtener vuelos de la fecha del último procesamiento guardada en el archivo pipeline.config.

Primer Fallback (API sin filtro): Si el primer intento falla (por un error de la API o por no encontrar datos para esa fecha), el script intenta una segunda llamada sin el filtro de fecha. Esto permite obtener los datos más recientes disponibles, aunque no se pueda garantizar la incrementalidad.

Segundo Fallback (Archivo de prueba): Si ambos intentos de la API fallan, el pipeline recurre a un archivo de prueba local (flights_raw_sample.csv). Este archivo contiene un pequeño conjunto de datos de ejemplo que asegura que el flujo de trabajo de carga y transformación pueda continuar y no se detenga por una falla en la fuente de datos.

Esta estrategia garantiza que el pipeline de datos sea más tolerante a fallos, permitiendo que las etapas posteriores (carga y transformación) puedan ejecutarse para fines de prueba, validación o cuando la disponibilidad de la API sea un problema.

In [2]:
!python ../src/extract.py

[2025-08-24 21:21:17] --- 🌍 Iniciando la etapa de Extracción ---
[2025-08-24 21:21:17] 🔎 Buscando vuelos desde la fecha: 2020-01-01
[2025-08-24 21:21:17] 🔗 Consultando API en el endpoint: flights
[2025-08-24 21:21:17] ❌ Error al consultar la API de vuelos: 403 Client Error: Forbidden for url: http://api.aviationstack.com/v1/flights?access_key=fee950e39aa10e46a6c8086f0cf579a2&limit=100&flight_date=2020-01-01
[2025-08-24 21:21:17] ⚠️ La API de vuelos no devolvió datos. Usando archivo de prueba...
[2025-08-24 21:21:17] ✅ Usando el archivo de prueba: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\processed\flights_raw_sample.csv
[2025-08-24 21:21:17] ✅ Columnas de vuelos renombradas para estandarizar el esquema.
[2025-08-24 21:21:17] ✅ Guardado DataFrame en c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\processed\flights_raw.parquet
[2025-08-24 21:21:17] 🔗 Consultando API en el endpoint: airports
[2025-08-24 21:21:18] ✅ Datos de 'airports' extraídos: 1

Verifico funcionalidad correcta de "Extract"

In [3]:
import pandas as pd
from pathlib import Path
# No se necesita redefinir las rutas, ya están en la celda de configuración inicial
# La lógica de validación ahora usa las variables globales.

if FLIGHTS_RAW_PARQUET.exists():
    df_flights = pd.read_parquet(FLIGHTS_RAW_PARQUET)
    print(f"✅ Filas en flights_raw.parquet: {len(df_flights)}")
else:
    print("❌ El archivo de vuelos no existe. Asegúrate de que la etapa de extracción se ejecutó correctamente.")

if AIRPORTS_RAW_PARQUET.exists():
    df_airports = pd.read_parquet(AIRPORTS_RAW_PARQUET)
    print(f"✅ Filas en airports_raw.parquet: {len(df_airports)}")
else:
    print("❌ El archivo de aeropuertos no existe. Asegúrate de que la etapa de extracción se ejecutó correctamente.")

NameError: name 'FLIGHTS_RAW_PARQUET' is not defined

## 2️⃣ Load

Ejecuta el script load.py.

Este script leerá los archivos Parquet.

Para los datos de aeropuertos, utilizará la estrategia full (hard_overwrite=True), borrando la tabla Delta existente y escribiendo el conjunto completo de datos estáticos.

Después de este paso, tu tabla Delta de aeropuertos contendrá todos los datos.

In [ ]:
!python ../src/load.py

## Extracción y Carga Incremental (Vuelos) 🛫
### Para simular una extracción incremental, debes correr el proceso una segunda vez.

Ejecuta extract.py y load.py nuevamente.

* extract.py: Leerá la fecha actualizada en pipeline.config y buscará nuevos datos desde esa fecha. El script actualizará la fecha en pipeline.config una vez que la extracción de vuelos sea exitosa.

* load.py: Leerá los nuevos datos de vuelos del archivo Parquet y, en lugar de borrar la tabla, los anexará a la tabla Delta de vuelos existente (mode="append").

### Verificación:

Abre pipeline.config y verás que la fecha de last_flights_run se actualizó a la fecha de la ejecución actual, demostrando que el proceso es stateful.

## 3️⃣ Transform

Ejecuta el script transform.py.

* Este script leerá todos los datos de la tabla Delta de vuelos (flights_processed_enriched), incluyendo los de la primera y segunda carga.

* Realizará la limpieza, el aplanamiento y el enriquecimiento de todo el historial acumulado.

In [ ]:
!python ../src/transform.py

## 4️⃣ Validación rápida de resultados 

Leemos las tablas Delta generadas y mostramos conteos básicos para asegurar que el pipeline corrió.


In [ ]:
# --- Funciones de ayuda ---
def get_parquet_row_count(path: Path) -> int:
    """Retorna el número de filas de un archivo Parquet si existe, de lo contrario 0."""
    if path.exists():
        try:
            return pd.read_parquet(path).shape[0]
        except Exception:
            return 0
    return 0

def get_delta_table_row_count(path: Path) -> int:
    """Retorna el número de filas de una tabla Delta si existe, de lo contrario 0."""
    try:
        if path.exists():
            return DeltaTable(str(path)).to_pandas().shape[0]
    except TableNotFoundError:
        return 0
    return 0

# --- Lógica principal ---
print("--- 📄 Resumen del Flujo de Datos ---")

# Recopilar métricas de cada etapa
summary = {
    "Etapa": ["Extract", "Extract", "Load", "Load", "Transform", "Transform"],
    "Descripción": [
        "Vuelos crudos (flights_raw.parquet)",
        "Aeropuertos crudos (airports_raw.parquet)",
        "Vuelos en Delta (flights/delta)",
        "Aeropuertos en Delta (airports/delta)",
        "Datos enriquecidos (processed_enriched)",
        "Datos agregados (agg_by_status)"
    ],
    "Recuento de Filas": [
        get_parquet_row_count(FLIGHTS_RAW_PARQUET),
        get_parquet_row_count(AIRPORTS_RAW_PARQUET),
        get_delta_table_row_count(FLIGHTS_DELTA_PATH),
        get_delta_table_row_count(AIRPORTS_DELTA_PATH),
        get_delta_table_row_count(PROCESSED_ENRICHED_PATH),
        get_delta_table_row_count(AGG_PATH)
    ],
    "Estado": ["❌ No encontrado", "❌ No encontrado", "❌ No encontrado", "❌ No encontrado", "❌ No encontrado", "❌ No encontrado"]
}

# Actualizar el estado basado en la existencia del archivo/tabla
for i, count in enumerate(summary["Recuento de Filas"]):
    if count > 0:
        summary["Estado"][i] = "✅ OK"

# Crear y mostrar el DataFrame de resumen
df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

print("\n--- ℹ️ Notas adicionales ---")
if not FLIGHTS_RAW_PARQUET.exists() or os.path.getsize(FLIGHTS_RAW_PARQUET) == 0:
    print("⚠️ El archivo de vuelos crudos no se generó o está vacío. El proceso de extracción puede haber fallado.")
if not AIRPORTS_RAW_PARQUET.exists() or os.path.getsize(AIRPORTS_RAW_PARQUET) == 0:
    print("⚠️ El archivo de aeropuertos crudos no se generó o está vacío. El proceso de extracción puede haber fallado.")
if not FLIGHTS_DELTA_PATH.exists() or get_delta_table_row_count(FLIGHTS_DELTA_PATH) == 0:
    print("⚠️ La tabla Delta de vuelos no se creó. La etapa de carga puede haber fallado.")
if not AIRPORTS_DELTA_PATH.exists() or get_delta_table_row_count(AIRPORTS_DELTA_PATH) == 0:
    print("⚠️ La tabla Delta de aeropuertos no se creó. La etapa de carga puede haber fallado.")
if not PROCESSED_ENRICHED_PATH.exists() or get_delta_table_row_count(PROCESSED_ENRICHED_PATH) == 0:
    print("⚠️ La tabla de datos enriquecidos no se creó. La etapa de transformación puede haber fallado.")
if not AGG_PATH.exists() or get_delta_table_row_count(AGG_PATH) == 0:
    print("⚠️ La tabla de datos agregados no se creó. La etapa de transformación de agregación puede haber fallado.")

## 5️⃣ Métricas y visualizaciones

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from deltalake import DeltaTable
from deltalake.exceptions import TableNotFoundError

# --- Rutas de las tablas Delta ---
DATA_LAKE_PATH = Path("../data_lake")
PROCESSED_ENRICHED_PATH = DATA_LAKE_PATH / "flights" / "processed_enriched"
AGG_PATH = DATA_LAKE_PATH / "flights" / "agg_by_status"

# --- Configuración de gráficos ---
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# --- Funciones de ayuda ---
def get_delta_table_as_df(path: Path) -> pd.DataFrame:
    """Carga una tabla Delta como un DataFrame de Pandas."""
    try:
        if path.exists():
            return DeltaTable(str(path)).to_pandas()
        else:
            print(f"⚠️ Advertencia: La tabla en la ruta {path} no se encuentra.")
            return pd.DataFrame()
    except TableNotFoundError:
        print(f"⚠️ Advertencia: No se pudo cargar la tabla Delta en {path}.")
        return pd.DataFrame()

# --- Cargar datos ---
print("Cargando datos para visualización...")
df_enriched = get_delta_table_as_df(PROCESSED_ENRICHED_PATH)
df_agg = get_delta_table_as_df(AGG_PATH)

# --- Generar gráficos si los DataFrames no están vacíos ---

if not df_agg.empty:
    print("Generando gráfico de barras: Total de vuelos por estado...")
    plt.figure(figsize=(10, 6))
    sns.barplot(x="status", y="total_flights", data=df_agg, palette="viridis")
    plt.title("Total de Vuelos por Estado", fontsize=16)
    plt.xlabel("Estado del Vuelo", fontsize=12)
    plt.ylabel("Total de Vuelos", fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    print("Gráfico de barras generado con éxito.")

if not df_agg.empty:
    print("Generando gráfico circular: Proporción de estados de vuelo...")
    plt.figure(figsize=(8, 8))
    plt.pie(
        df_agg["total_flights"],
        labels=df_agg["status"],
        autopct="%1.1f%%",
        startangle=140,
        colors=sns.color_palette("viridis", n_colors=len(df_agg))
    )
    plt.title("Proporción de Estados de Vuelo", fontsize=16)
    plt.show()
    print("Gráfico circular generado con éxito.")

if not df_enriched.empty:
    print("Generando gráfico de líneas: Vuelos diarios...")
    df_flights_by_date = df_enriched.groupby("flight_date").size().reset_index(name="count")
    
    plt.figure(figsize=(12, 6))
    sns.lineplot(x="flight_date", y="count", data=df_flights_by_date, marker="o")
    plt.title("Vuelos Diarios", fontsize=16)
    plt.xlabel("Fecha", fontsize=12)
    plt.ylabel("Número de Vuelos", fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    print("Gráfico de líneas generado con éxito.")

if df_agg.empty and df_enriched.empty:
    print("❌ No se pudieron generar gráficos. Asegúrate de que las etapas de carga y transformación se ejecutaron correctamente y que los archivos Delta no están vacíos.")